# C2_supp supplementary experiments on Colab (CPU runtime)
Run the cells in order. After a disconnect: rerun cells 1, 2 (unzip is skipped if `/content/C2_supp` exists), 3, then 5; finished tasks are skipped.
If Drive cannot be mounted, cell 1 falls back to the local disk `/content/C2_run`; then pull `C2_run/C2_partial_latest.zip` from the host side regularly (it is rewritten every 25 tasks / 15 min), because a runtime reset deletes the local disk.

In [ ]:
# 1. Output location: Google Drive if it can be mounted, otherwise local disk (host-side pulls of the checkpoint zip)
import os
RUN = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN = '/content/drive/MyDrive/C2_supp_run'
except Exception as e:
    print('Drive not available (%r): using local disk /content/C2_run; pull C2_run/C2_partial_latest.zip regularly' % (e,))
    RUN = '/content/C2_run'
OUT = RUN + '/C2_out'
os.makedirs(RUN, exist_ok=True); print('RUN =', RUN, ' OUT =', OUT)

In [ ]:
# 2. Unzip the uploaded package (upload C2_supp_upload.zip to /content or to RUN first)
import os, zipfile
src = '/content/C2_supp_upload.zip' if os.path.exists('/content/C2_supp_upload.zip') else RUN + '/C2_supp_upload.zip'
if not os.path.exists('/content/C2_supp'):
    zipfile.ZipFile(src).extractall('/content')
%cd /content/C2_supp
!ls

In [ ]:
# 3. Install and record the environment
!pip -q install -r requirements.txt
!python run_c2.py env --out {OUT}

In [ ]:
# 4. Sanity + smoke (must end with SMOKE PASS; otherwise stop and report)
import subprocess
subprocess.run(["python", "run_c2.py", "smoke"], check=True)

In [ ]:
# 5. All tasks in ONE foreground process (sequential, so timings are comparable). Resume = rerun this cell.
#    Order: LASSO references first, then a_struct, a_range, b. Log: OUT/logs/run_log.txt
import subprocess
r = subprocess.run(['python', 'run_c2.py', 'run', '--group', 'all', '--out', OUT])
print('run exit code', r.returncode, '(1 = some tasks failed: see OUT/logs/FAILED__*.txt, report them)')

In [ ]:
# 6. Progress
!python run_c2.py status --out {OUT}
!tail -n 5 {OUT}/logs/run_log.txt

In [ ]:
# 7. (optional) partial checkpoint now -> RUN/C2_partial_latest.zip
!python run_c2.py checkpoint --out {OUT}

In [ ]:
# 8. Pack (only after status shows all tasks done). Exit code 2 = missing tasks or strict plot failure: report it.
!python run_c2.py pack --out {OUT}; echo exit=$?
!cp -v /content/C2_supp/C2_results_*.zip {RUN}/

In [ ]:
# 9. Download
import glob
from google.colab import files
z = sorted(glob.glob('/content/C2_supp/C2_results_*.zip'))[-1]
print(z); files.download(z)